# 06 — Production Scoring & Prediction Pipeline

## 1. Operational Purpose & Core Architecture
This notebook implements the **Production Scoring & Prediction Pipeline**. It loads the verified frozen LightGBM booster produced by `04_lightgbm_model.ipynb` and verified by `05_model_diagnostics_and_error_analysis.ipynb`, scores incoming feature store vectors for $D \rightarrow D+1$ demand forecasting, applies calibrated empirical uncertainty intervals, and persists an immutable, partitioned prediction dataset for downstream inventory optimization (`07_inventory_optimization.ipynb`).

```
┌─────────────────────────┐     ┌─────────────────────────┐     ┌─────────────────────────┐
│ Notebook 04             │     │ Notebook 05             │     │ Notebook 06             │
│ Frozen LightGBM Booster │────►│ Diagnostic Readiness    │────►│ Production Scoring      │
│ (Run: e8da7be6...)      │     │ (Gating: 0 Blockers)    │     │ (Partitioned Parquet)   │
└─────────────────────────┘     └─────────────────────────┘     └────────────┬────────────┘
                                                                             │
                                                                             ▼
                                                                ┌─────────────────────────┐
                                                                │ Notebook 07             │
                                                                │ Newsvendor Replenishment│
                                                                │ (Optimal Orders Q*)     │
                                                                └─────────────────────────┘
```

### 2. Dual Scoring Horizons
This pipeline supports two distinct scoring operational scopes:
- **Mode A (`HOLDOUT_REPLAY`):** Historical scoring across the full holdout horizon (`forecast_date` from `2016-12-31` to `2017-08-14` $\rightarrow$ `target_date` `2017-01-01` to `2017-08-15`, 19,571,966 rows across 227 dates) saved in partitioned Parquet format.
- **Mode B (`OPERATIONAL_SIMULATION`):** Operational day simulation scoring the latest available historical origin ($D = \text{2017-08-14} \rightarrow D+1 = \text{2017-08-15}$, 84,659 active store-SKUs across 54 stores) saved as a standalone operational forecast.

### 3. Strict Frozen-Model Governance Invariants
- **ZERO Retraining / ZERO Tuning:** Booster model is executed strictly in read-only inference mode.
- **Zero Target Leakage:** The target column `target_unit_sales` is **never selected** during inference.
- **D $\rightarrow$ D+1 Invariant:** `target_date == forecast_date + 1 day` is strictly enforced with zero violations.


In [1]:
# Cell 2 — Setup, Dependencies, Governance Sentinels & Path Discovery
import os
import sys
import gc
import json
import uuid
import sqlite3
import hashlib
from datetime import datetime, timezone
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

import duckdb
import numpy as np
import pandas as pd
import lightgbm as lgb

print(f"DuckDB version   : {duckdb.__version__}")
print(f"LightGBM version : {lgb.__version__}")
print(f"Pandas version   : {pd.__version__}")

# ── 1. Governance Sentinels (NEVER MUTATED) ──────────────────────────────────
RETRAINED = False
RETUNED = False
MODEL_SELECTION_PERFORMED = False
PRODUCTION_SCORING_RUN = True
RANDOM_SEED = 42

SCORING_RUN_ID = str(uuid.uuid4())
SCORING_TIMESTAMP = datetime.now(timezone.utc).isoformat()

print(f"\n[GOVERNANCE] Scoring Run Initialized:")
print(f"  Scoring Run ID            : {SCORING_RUN_ID}")
print(f"  Scoring Timestamp (UTC)   : {SCORING_TIMESTAMP}")
print(f"  RETRAINED                 : {RETRAINED}")
print(f"  RETUNED                   : {RETUNED}")
print(f"  MODEL_SELECTION_PERFORMED : {MODEL_SELECTION_PERFORMED}")
print(f"  PRODUCTION_SCORING_RUN    : {PRODUCTION_SCORING_RUN}")

# ── 2. Lakehouse Path Resolution ─────────────────────────────────────────────
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

GOLD_DIR      = lakehouse_root / "data_lake" / "gold"
FEATURES_DIR  = GOLD_DIR / "gold_forecasting_features"
BASELINE_DIR  = GOLD_DIR / "forecasting_baseline_results"
LGBM_DIR      = GOLD_DIR / "lightgbm_results"
DIAG_DIR      = GOLD_DIR / "lightgbm_diagnostics"
PROD_PRED_DIR = GOLD_DIR / "production_predictions"
PROD_PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nLakehouse root         : {lakehouse_root}")
print(f"Features directory     : {FEATURES_DIR}")
print(f"Diagnostics directory  : {DIAG_DIR}")
print(f"Production Predictions : {PROD_PRED_DIR}")

# ── 3. Prerequisite File Assertions ──────────────────────────────────────────
required_inputs = [
    LGBM_DIR / "model_contract.json",
    LGBM_DIR / "lightgbm_model.txt",
    DIAG_DIR / "diagnostic_report.json",
    DIAG_DIR / "demand_tier_metrics.parquet",
    DIAG_DIR / "diagnostic_predictions_sample.parquet",
]
for p in required_inputs:
    assert p.exists(), f"[FAIL] Required input artifact missing: {p}"
    print(f"[OK] Found input artifact: {p.name} ({p.stat().st_size:,} bytes)")


DuckDB version   : 1.4.5
LightGBM version : 4.6.0
Pandas version   : 2.3.2

[GOVERNANCE] Scoring Run Initialized:
  Scoring Run ID            : 50d6c4cf-6d07-42e9-a72b-7192e2f3268d
  Scoring Timestamp (UTC)   : 2026-08-20T10:46:57.607587+00:00
  RETRAINED                 : False
  RETUNED                   : False
  MODEL_SELECTION_PERFORMED : False
  PRODUCTION_SCORING_RUN    : True

Lakehouse root         : D:\Demand_forecasting\inventory-lakehouse
Features directory     : D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\gold_forecasting_features
Diagnostics directory  : D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\lightgbm_diagnostics
Production Predictions : D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\production_predictions
[OK] Found input artifact: model_contract.json (3,261 bytes)
[OK] Found input artifact: lightgbm_model.txt (5,661,545 bytes)
[OK] Found input artifact: diagnostic_report.json (2,387 bytes)
[OK] Found input artifact: demand_tier_m

In [2]:
# Cell 3 — Upstream Production Readiness Gate (Consuming NB05 Diagnostic Report)
DIAG_REPORT_PATH = DIAG_DIR / "diagnostic_report.json"
with open(DIAG_REPORT_PATH) as f:
    diag_report = json.load(f)

print("[CHECK] Evaluating Notebook 05 Upstream Readiness Gate...")

# 1. Hard Technical Blocker Gate Assertions
assert diag_report["diagnostic_conclusion"]["production_block"] is False, \
    f"[FATAL] NB05 blocked production scoring! Blockers: {diag_report.get('blocking_issues')}"
assert len(diag_report["blocking_issues"]) == 0, \
    f"[FATAL] Blocking issues detected in diagnostic report: {diag_report['blocking_issues']}"
assert diag_report["governance"]["retrained"] is False, "[FATAL] Model was illegally retrained"
assert diag_report["governance"]["retuned"] is False, "[FATAL] Model was illegally retuned"
assert diag_report["governance"]["model_selection_performed"] is False
assert diag_report["governance"]["test_result_modified"] is False

print(f"[PASS] Hard Technical Blocker Gate Passed (0 blocking issues).")

# 2. Informational Investigation Warnings (Non-Blocking)
investigation_items = diag_report.get("investigation_items", [])
if len(investigation_items) > 0:
    print(f"\n[INFO] Upstream Investigation Warnings ({len(investigation_items)} items logged):")
    for item in investigation_items:
        print(f"  - [WARNING] {item}")
else:
    print("\n[INFO] Zero upstream investigation warnings.")


[CHECK] Evaluating Notebook 05 Upstream Readiness Gate...
[PASS] Hard Technical Blocker Gate Passed (0 blocking issues).

[INFO] Upstream Investigation Warnings (2 items logged):
  - [WARNING] 1M Test Diagnostic Sample WMAPE is 0.4821 (official 10M test population is 0.4492) — reflects early Jan sample slice volatility
  - [WARNING] Distribution shift observed in features: ['oil_price', 'on_promotion_tomorrow']


In [3]:
# Cell 4 — Complete 64-Character SHA-256 Checksum & Booster Loading
MODEL_PATH = LGBM_DIR / "lightgbm_model.txt"

def compute_sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_SHA256 = "ea408d13e01a64f51ddd73554dfe15b7206dc1be13142970734507112ae4a40f"
ACTUAL_SHA256 = compute_sha256(MODEL_PATH)

print(f"Expected Model SHA-256 : {EXPECTED_SHA256}")
print(f"Actual Model SHA-256   : {ACTUAL_SHA256}")
assert ACTUAL_SHA256 == EXPECTED_SHA256 == diag_report["model_sha256"], \
    f"[FATAL] Cryptographic checksum mismatch for {MODEL_PATH.name}!"

# Load native booster
booster = lgb.Booster(model_file=str(MODEL_PATH))

assert isinstance(booster, lgb.Booster), "[FAIL] Loaded object is not a LightGBM Booster"
assert booster.num_feature() == 24, f"[FAIL] Expected 24 features in booster, found {booster.num_feature()}"
assert booster.num_trees() == 496, f"[FAIL] Expected 496 trees in booster, found {booster.num_trees()}"

print(f"\n[PASS] Bit-Identical Frozen Booster Model Loaded:")
print(f"       File Size     : {MODEL_PATH.stat().st_size:,} bytes")
print(f"       Features      : {booster.num_feature()}")
print(f"       Trees         : {booster.num_trees()}")


Expected Model SHA-256 : ea408d13e01a64f51ddd73554dfe15b7206dc1be13142970734507112ae4a40f
Actual Model SHA-256   : ea408d13e01a64f51ddd73554dfe15b7206dc1be13142970734507112ae4a40f

[PASS] Bit-Identical Frozen Booster Model Loaded:
       File Size     : 5,661,545 bytes
       Features      : 24
       Trees         : 496


In [4]:
# Cell 5 — Dynamic Feature Contract & 4-Tier Order Chain Validation
MODEL_CONTRACT_PATH = LGBM_DIR / "model_contract.json"
with open(MODEL_CONTRACT_PATH) as f:
    mc = json.load(f)

# Dynamically extract feature definitions
FEATURES = mc["model"]["features"]
CATEGORICAL_FEATURES = mc["model"]["categorical_features"]
NUMERIC_FEATURES = [f for f in FEATURES if f not in CATEGORICAL_FEATURES]
IDENTIFIERS = ["forecast_date", "target_date", "store_id", "sku_id"]
FINAL_MODEL_RUN_ID = mc["mlflow"]["final_model_run_id"]

# Validate 4-Tier Feature Order Chain
assert len(FEATURES) == 24, f"[FAIL] Expected 24 features, got {len(FEATURES)}"
assert list(booster.feature_name()) == FEATURES, "[FATAL] Booster feature order does not match contract"
assert mc["model"]["features"] == FEATURES, "[FATAL] Contract feature definition mismatch"
assert "target_unit_sales" not in FEATURES, "[FATAL] Target leaked into feature list"
assert set(CATEGORICAL_FEATURES).issubset(set(FEATURES)), "[FATAL] Categoricals not subset of features"

print("[PASS] 4-Tier Feature Order Chain Verified:")
print(f"       Total Features       : {len(FEATURES)}")
print(f"       Categorical Features : {len(CATEGORICAL_FEATURES)} -> {CATEGORICAL_FEATURES}")
print(f"       Numeric Features     : {len(NUMERIC_FEATURES)}")
print(f"       Model Run ID         : {FINAL_MODEL_RUN_ID}")


[PASS] 4-Tier Feature Order Chain Verified:
       Total Features       : 24
       Categorical Features : 4 -> ['family', 'city', 'store_type', 'cluster']
       Numeric Features     : 20
       Model Run ID         : e8da7be6df9e41e9b3ef197afbe08315


In [5]:
# Cell 6 — Empirical Demand-Tier Residual Calibration Setup (Validation-Derived)
q33 = float(diag_report["demand_tier_thresholds"]["q33"])  # 2.785714
q66 = float(diag_report["demand_tier_thresholds"]["q66"])  # 6.964286

# Empirical residual quantiles (q10 and q90) and expected MAE by demand tier
# Computed strictly from the frozen validation dataset residuals (actual - prediction)
TIER_CALIBRATION = {
    "1_Low": {
        "q10": -2.109779,
        "q90": 2.330877,
        "expected_mae": 1.643431,
        "description": "Low demand tier (rolling_mean_28 <= 2.7857)"
    },
    "2_Mid": {
        "q10": -3.960390,
        "q90": 4.383517,
        "expected_mae": 2.926671,
        "description": "Mid demand tier (2.7857 < rolling_mean_28 <= 6.9643)"
    },
    "3_High": {
        "q10": -10.786951,
        "q90": 10.365351,
        "expected_mae": 8.258449,
        "description": "High demand tier (rolling_mean_28 > 6.9643)"
    }
}

calib_summary_df = pd.DataFrame.from_dict(TIER_CALIBRATION, orient="index")
print("[OK] Empirical Demand-Tier Residual Calibration Offsets (Frozen from Validation Set):")
display(calib_summary_df)

print("\n[NOTE] Uncertainty Bounds Formulation:")
print("       operational_lower_80 = max(0.0, predicted_unit_sales + q10_tier)")
print("       operational_upper_80 = predicted_unit_sales + q90_tier")
print("       Provides empirical 80% coverage without parametric normal distribution assumptions.")


[OK] Empirical Demand-Tier Residual Calibration Offsets (Frozen from Validation Set):


,q10,q90,expected_mae,description
1_Low,-2.109779,2.330877,1.643431,Low demand tier (rolling_mean_28 <= 2.7857)
2_Mid,-3.960390,4.383517,2.926671,Mid demand tier (2.7857 < rolling_mean_28 <= 6...
3_High,-10.786951,10.365351,8.258449,High demand tier (rolling_mean_28 > 6.9643)



[NOTE] Uncertainty Bounds Formulation:
       operational_lower_80 = max(0.0, predicted_unit_sales + q10_tier)
       operational_upper_80 = predicted_unit_sales + q90_tier
       Provides empirical 80% coverage without parametric normal distribution assumptions.


In [6]:
# Cell 7 — DuckDB Streaming Engine & Strict Invariant Validation
con = duckdb.connect()
glob_pattern = str(FEATURES_DIR / "*.parquet").replace("\\", "/")
con.execute(f"CREATE OR REPLACE VIEW features AS SELECT * FROM read_parquet('{glob_pattern}')")

# Create Horizons
con.execute("""
    CREATE OR REPLACE VIEW holdout_replay AS 
    SELECT * FROM features 
    WHERE target_date >= '2017-01-01' AND target_date <= '2017-08-15'
""")

con.execute("""
    CREATE OR REPLACE VIEW operational_simulation AS 
    SELECT * FROM features 
    WHERE forecast_date = '2017-08-14'
""")

# 1. Strict D -> D+1 Invariant Assertion (Zero violations allowed)
d_violations = con.execute("""
    SELECT COUNT(*) FROM holdout_replay 
    WHERE target_date != forecast_date + INTERVAL 1 DAY
""").fetchone()[0]
assert d_violations == 0, f"[FATAL] D -> D+1 invariant violated in {d_violations} rows!"

# 2. Input Key Uniqueness Assertion
dup_input_keys = con.execute("""
    SELECT COUNT(*) - COUNT(DISTINCT store_id || '|' || sku_id || '|' || CAST(forecast_date AS VARCHAR))
    FROM holdout_replay
""").fetchone()[0]
assert dup_input_keys == 0, f"[FATAL] Duplicate keys in holdout input: {dup_input_keys}"

# 3. Completeness & Null Key Check
null_keys = con.execute("""
    SELECT SUM(CASE WHEN store_id IS NULL OR sku_id IS NULL OR forecast_date IS NULL OR target_date IS NULL THEN 1 ELSE 0 END)
    FROM holdout_replay
""").fetchone()[0]
assert null_keys == 0, f"[FATAL] Null keys detected: {null_keys}"

total_holdout_rows = con.execute("SELECT COUNT(*) FROM holdout_replay").fetchone()[0]
n_forecast_dates = con.execute("SELECT COUNT(DISTINCT forecast_date) FROM holdout_replay").fetchone()[0]
operational_rows = con.execute("SELECT COUNT(*) FROM operational_simulation").fetchone()[0]

print(f"[PASS] Invariant Validation Passed:")
print(f"       Holdout Replay Rows    : {total_holdout_rows:,} across {n_forecast_dates} dates (2016-12-31 to 2017-08-14)")
print(f"       Operational Slice Rows : {operational_rows:,} (forecast_date: 2017-08-14 -> target_date: 2017-08-15)")
print(f"       D -> D+1 Violations    : 0")
print(f"       Duplicate Input Keys   : 0")


[PASS] Invariant Validation Passed:
       Holdout Replay Rows    : 19,571,966 across 227 dates (2016-12-31 to 2017-08-14)
       Operational Slice Rows : 84,659 (forecast_date: 2017-08-14 -> target_date: 2017-08-15)
       D -> D+1 Violations    : 0
       Duplicate Input Keys   : 0


In [7]:
# Cell 8 — Mandatory 1,000-Row Deterministic Smoke Test
smoke_cols = ", ".join(IDENTIFIERS + ["family", "rolling_mean_28"] + FEATURES)
smoke_df = con.execute(f"""
    SELECT {smoke_cols}
    FROM holdout_replay
    USING SAMPLE 1000 (reservoir, {RANDOM_SEED})
""").df()

for col in CATEGORICAL_FEATURES:
    smoke_df[col] = smoke_df[col].astype("category")
for col in NUMERIC_FEATURES:
    smoke_df[col] = smoke_df[col].astype("float32")

smoke_preds = booster.predict(smoke_df[FEATURES])

# Assertions on smoke predictions
assert not np.any(np.isnan(smoke_preds)), "[FATAL] NaN detected in smoke predictions"
assert not np.any(np.isinf(smoke_preds)), "[FATAL] Inf detected in smoke predictions"
assert len(smoke_preds) == 1000, "[FATAL] Smoke prediction row count mismatch"
assert np.issubdtype(smoke_preds.dtype, np.floating), "[FATAL] Predictions not floating-point"

print(f"[PASS] Mandatory Deterministic Smoke Test Passed (1,000 rows scored cleanly).")
print(f"       Min Smoke Pred : {np.min(smoke_preds):.4f}")
print(f"       Mean Smoke Pred: {np.mean(smoke_preds):.4f}")
print(f"       Max Smoke Pred : {np.max(smoke_preds):.4f}")


[PASS] Mandatory Deterministic Smoke Test Passed (1,000 rows scored cleanly).
       Min Smoke Pred : 1.4798
       Mean Smoke Pred: 9.0110
       Max Smoke Pred : 324.4859


In [8]:
# Cell 9 — Core Batch Scoring & Post-Processing Function
OUTPUT_COLUMNS = [
    "forecast_date", "target_date", "store_id", "sku_id", "family",
    "raw_prediction", "predicted_unit_sales", "demand_tier",
    "expected_error_mae", "operational_lower_80", "operational_upper_80",
    "model_run_id", "scoring_run_id", "scoring_timestamp"
]

def score_batch(df_chunk: pd.DataFrame, scoring_run_id: str, scoring_timestamp: str) -> pd.DataFrame:
    # 1. Strict Feature Casting (Zero mutation of original feature vectors)
    for col in CATEGORICAL_FEATURES:
        df_chunk[col] = df_chunk[col].astype("category")
    for col in NUMERIC_FEATURES:
        df_chunk[col] = df_chunk[col].astype("float32")
        
    # 2. Native LightGBM Booster Inference
    raw_preds = booster.predict(df_chunk[FEATURES])
    df_chunk["raw_prediction"] = raw_preds.astype("float32")
    
    # 3. Operational Point Forecast (Clipping at >= 0.0)
    df_chunk["predicted_unit_sales"] = np.clip(raw_preds, 0.0, None).astype("float32")
    
    # 4. Demand Tier Assignment (Train thresholds)
    rm28 = df_chunk["rolling_mean_28"].values
    tier_arr = np.where(rm28 <= q33, "1_Low", np.where(rm28 <= q66, "2_Mid", "3_High"))
    df_chunk["demand_tier"] = tier_arr
    
    # 5. Calibrated Empirical Uncertainty Bounds
    q10_arr = np.where(tier_arr == "1_Low", TIER_CALIBRATION["1_Low"]["q10"],
              np.where(tier_arr == "2_Mid", TIER_CALIBRATION["2_Mid"]["q10"], TIER_CALIBRATION["3_High"]["q10"]))
    q90_arr = np.where(tier_arr == "1_Low", TIER_CALIBRATION["1_Low"]["q90"],
              np.where(tier_arr == "2_Mid", TIER_CALIBRATION["2_Mid"]["q90"], TIER_CALIBRATION["3_High"]["q90"]))
    mae_arr = np.where(tier_arr == "1_Low", TIER_CALIBRATION["1_Low"]["expected_mae"],
              np.where(tier_arr == "2_Mid", TIER_CALIBRATION["2_Mid"]["expected_mae"], TIER_CALIBRATION["3_High"]["expected_mae"]))
    
    df_chunk["operational_lower_80"] = np.clip(df_chunk["predicted_unit_sales"] + q10_arr, 0.0, None).astype("float32")
    df_chunk["operational_upper_80"] = (df_chunk["predicted_unit_sales"] + q90_arr).astype("float32")
    df_chunk["expected_error_mae"] = mae_arr.astype("float32")
    
    # 6. Provenance & Run Metadata
    df_chunk["model_run_id"] = FINAL_MODEL_RUN_ID
    df_chunk["scoring_run_id"] = scoring_run_id
    df_chunk["scoring_timestamp"] = scoring_timestamp
    
    return df_chunk[OUTPUT_COLUMNS]

print("[OK] Production Batch Scoring Function `score_batch` initialized.")


[OK] Production Batch Scoring Function `score_batch` initialized.


In [9]:
# Cell 10 — Mode B: Execute Operational Simulation Scoring (D = 2017-08-14)
print("[RUN] Executing Mode B: Operational Simulation Scoring for latest origin (2017-08-14)...")

query_cols = ", ".join(IDENTIFIERS + ["family", "rolling_mean_28"] + FEATURES)
op_input_df = con.execute(f"""
    SELECT {query_cols}
    FROM operational_simulation
""").df()

op_scored_df = score_batch(op_input_df, SCORING_RUN_ID, SCORING_TIMESTAMP)

# Invariant Assertions on Operational Slice
assert len(op_scored_df) == 84659, f"[FAIL] Expected 84,659 rows, got {len(op_scored_df):,}"
assert op_scored_df["forecast_date"].nunique() == 1, "[FAIL] Multi-forecast dates in operational slice"
assert op_scored_df["target_date"].nunique() == 1, "[FAIL] Multi-target dates in operational slice"
assert op_scored_df[["store_id", "sku_id", "target_date"]].duplicated().sum() == 0, "[FAIL] Duplicate keys"
assert (op_scored_df["predicted_unit_sales"] < 0).sum() == 0, "[FAIL] Negative predictions in operational slice"
assert (op_scored_df["operational_lower_80"] > op_scored_df["predicted_unit_sales"]).sum() == 0
assert (op_scored_df["operational_upper_80"] < op_scored_df["predicted_unit_sales"]).sum() == 0

# Save standalone operational slice
OP_FILE_PATH = PROD_PRED_DIR / "operational_simulation_latest.parquet"
op_scored_df.to_parquet(OP_FILE_PATH, index=False)

print(f"[PASS] Mode B Operational Simulation Complete:")
print(f"       Saved File      : {OP_FILE_PATH}")
print(f"       File Size       : {OP_FILE_PATH.stat().st_size:,} bytes")
print(f"       Total Store-SKUs: {len(op_scored_df):,}")
print(f"       Forecast Date   : {op_scored_df['forecast_date'].iloc[0]}")
print(f"       Target Date     : {op_scored_df['target_date'].iloc[0]}")

print("\nTop 5 Operational Forecast Rows Preview:")
display(op_scored_df.head(5))


[RUN] Executing Mode B: Operational Simulation Scoring for latest origin (2017-08-14)...


[PASS] Mode B Operational Simulation Complete:
       Saved File      : D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\production_predictions\operational_simulation_latest.parquet
       File Size       : 2,161,201 bytes
       Total Store-SKUs: 84,659
       Forecast Date   : 2017-08-14 00:00:00
       Target Date     : 2017-08-15 00:00:00

Top 5 Operational Forecast Rows Preview:


,forecast_date,target_date,store_id,sku_id,family,raw_prediction,predicted_unit_sales,demand_tier,expected_error_mae,operational_lower_80,operational_upper_80,model_run_id,scoring_run_id,scoring_timestamp
0,2017-08-14,2017-08-15,1,103665,BREAD/BAKERY,3.327362,3.327362,1_Low,1.643431,1.217583,5.658239,e8da7be6df9e41e9b3ef197afbe08315,50d6c4cf-6d07-42e9-a72b-7192e2f3268d,2026-08-20T10:46:57.607587+00:00
1,2017-08-14,2017-08-15,1,105574,GROCERY I,4.616930,4.616930,2_Mid,2.926671,0.656540,9.000447,e8da7be6df9e41e9b3ef197afbe08315,50d6c4cf-6d07-42e9-a72b-7192e2f3268d,2026-08-20T10:46:57.607587+00:00
2,2017-08-14,2017-08-15,1,105575,GROCERY I,10.597316,10.597316,3_High,8.258449,0.000000,20.962667,e8da7be6df9e41e9b3ef197afbe08315,50d6c4cf-6d07-42e9-a72b-7192e2f3268d,2026-08-20T10:46:57.607587+00:00
3,2017-08-14,2017-08-15,1,105577,GROCERY I,1.884764,1.884764,1_Low,1.643431,0.000000,4.215641,e8da7be6df9e41e9b3ef197afbe08315,50d6c4cf-6d07-42e9-a72b-7192e2f3268d,2026-08-20T10:46:57.607587+00:00
4,2017-08-14,2017-08-15,1,105693,GROCERY I,1.833604,1.833604,1_Low,1.643431,0.000000,4.164481,e8da7be6df9e41e9b3ef197afbe08315,50d6c4cf-6d07-42e9-a72b-7192e2f3268d,2026-08-20T10:46:57.607587+00:00


In [10]:
# Cell 11 — Mode A: Execute Full Holdout Replay Partitioned Scoring (227 Dates)
print("[RUN] Executing Mode A: Full Holdout Replay Partitioned Scoring (19.57M rows across 227 dates)...")

forecast_dates_list = con.execute("""
    SELECT DISTINCT forecast_date 
    FROM holdout_replay 
    ORDER BY forecast_date
""").df()["forecast_date"].tolist()

print(f"Total dates to process: {len(forecast_dates_list)} (from {forecast_dates_list[0]} to {forecast_dates_list[-1]})")

manifest_records = []
total_rows_scored = 0
start_time_all = datetime.now()

query_cols = ", ".join(IDENTIFIERS + ["family", "rolling_mean_28"] + FEATURES)

for i, fc_date in enumerate(forecast_dates_list, 1):
    date_str = str(pd.to_datetime(fc_date).date())
    
    # Extract partition using DuckDB
    df_date = con.execute(f"""
        SELECT {query_cols}
        FROM holdout_replay
        WHERE forecast_date = '{fc_date}'
    """).df()
    
    # Score partition
    scored_date_df = score_batch(df_date, SCORING_RUN_ID, SCORING_TIMESTAMP)
    
    # Per-partition invariant assertions
    assert len(scored_date_df) == len(df_date), f"[FAIL] Row count mismatch on {date_str}"
    assert scored_date_df["forecast_date"].nunique() == 1
    assert scored_date_df["target_date"].nunique() == 1
    assert scored_date_df[["store_id", "sku_id", "target_date"]].duplicated().sum() == 0
    assert (scored_date_df["predicted_unit_sales"] < 0).sum() == 0
    
    # Write partitioned Parquet
    part_dir = PROD_PRED_DIR / f"forecast_date={date_str}"
    part_dir.mkdir(parents=True, exist_ok=True)
    part_file = part_dir / "part-00000.parquet"
    scored_date_df.to_parquet(part_file, index=False)
    
    total_rows_scored += len(scored_date_df)
    
    manifest_records.append({
        "forecast_date": date_str,
        "target_date": str(pd.to_datetime(scored_date_df["target_date"].iloc[0]).date()),
        "row_count": len(scored_date_df),
        "partition_path": str(part_file),
        "file_size_bytes": part_file.stat().st_size
    })
    
    if i % 30 == 0 or i == len(forecast_dates_list):
        elapsed_sec = (datetime.now() - start_time_all).total_seconds()
        rate = total_rows_scored / max(elapsed_sec, 0.001)
        print(f"  Processed {i:>3}/{len(forecast_dates_list)} dates ({total_rows_scored:>10,} rows) — {rate:>8,.0f} rows/sec")
    
    del df_date, scored_date_df
    if i % 20 == 0:
        gc.collect()

print(f"\n[PASS] Mode A Holdout Replay Complete: {total_rows_scored:,} rows written across {len(manifest_records)} partitions.")


[RUN] Executing Mode A: Full Holdout Replay Partitioned Scoring (19.57M rows across 227 dates)...
Total dates to process: 227 (from 2016-12-31 00:00:00 to 2017-08-14 00:00:00)


  Processed  30/227 dates ( 2,429,815 rows) —   62,868 rows/sec


  Processed  60/227 dates ( 5,042,685 rows) —   61,008 rows/sec


  Processed  90/227 dates ( 7,647,013 rows) —   60,951 rows/sec


  Processed 120/227 dates (10,234,226 rows) —   63,413 rows/sec


  Processed 150/227 dates (12,841,184 rows) —   64,978 rows/sec


  Processed 180/227 dates (15,459,488 rows) —   65,988 rows/sec


  Processed 210/227 dates (18,080,565 rows) —   60,072 rows/sec


  Processed 227/227 dates (19,571,966 rows) —   58,335 rows/sec

[PASS] Mode A Holdout Replay Complete: 19,571,966 rows written across 227 partitions.


In [11]:
# Cell 12 — Hard Quality Gate & Mathematical Consistency Verification
print("[CHECK] Executing Hard Quality Gate on Scored Datasets...")

parquet_glob = str(PROD_PRED_DIR / "forecast_date=*/part-*.parquet").replace("\\", "/")

# Verify overall holdout replay totals via DuckDB
total_partition_rows = con.execute(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT forecast_date) AS n_fc_dates,
        COUNT(DISTINCT target_date) AS n_tgt_dates,
        SUM(CASE WHEN predicted_unit_sales < 0 THEN 1 ELSE 0 END) AS negative_preds,
        SUM(CASE WHEN operational_lower_80 > predicted_unit_sales THEN 1 ELSE 0 END) AS lower_bound_violations,
        SUM(CASE WHEN operational_upper_80 < predicted_unit_sales THEN 1 ELSE 0 END) AS upper_bound_violations,
        SUM(CASE WHEN predicted_unit_sales IS NULL THEN 1 ELSE 0 END) AS null_preds
    FROM read_parquet('{parquet_glob}')
""").df()

display(total_partition_rows)

assert total_partition_rows["total_rows"].iloc[0] == 19571966, f"[FATAL] Total row mismatch: {total_partition_rows['total_rows'].iloc[0]:,}"
assert total_partition_rows["n_fc_dates"].iloc[0] == 227, "[FATAL] Partition count mismatch"
assert total_partition_rows["negative_preds"].iloc[0] == 0, "[FATAL] Negative predictions detected"
assert total_partition_rows["lower_bound_violations"].iloc[0] == 0, "[FATAL] Lower bound > point forecast"
assert total_partition_rows["upper_bound_violations"].iloc[0] == 0, "[FATAL] Upper bound < point forecast"
assert total_partition_rows["null_preds"].iloc[0] == 0, "[FATAL] Null predictions detected"

print("[PASS] Hard Quality Gate Passed: 100% mathematical consistency and key integrity verified.")


[CHECK] Executing Hard Quality Gate on Scored Datasets...


,total_rows,n_fc_dates,n_tgt_dates,negative_preds,lower_bound_violations,upper_bound_violations,null_preds
0,19571966,227,227,0.0,0.0,0.0,0.0


[PASS] Hard Quality Gate Passed: 100% mathematical consistency and key integrity verified.


In [12]:
# Cell 13 — Forecast Distribution & Volume Sanity Check
parquet_glob = str(PROD_PRED_DIR / "forecast_date=*/part-*.parquet").replace("\\", "/")

dist_stats = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(raw_prediction) AS min_raw,
        MIN(predicted_unit_sales) AS min_clipped,
        QUANTILE_CONT(predicted_unit_sales, 0.05) AS p5,
        QUANTILE_CONT(predicted_unit_sales, 0.25) AS p25,
        QUANTILE_CONT(predicted_unit_sales, 0.50) AS median,
        AVG(predicted_unit_sales) AS mean,
        QUANTILE_CONT(predicted_unit_sales, 0.75) AS p75,
        QUANTILE_CONT(predicted_unit_sales, 0.95) AS p95,
        QUANTILE_CONT(predicted_unit_sales, 0.99) AS p99,
        MAX(predicted_unit_sales) AS max,
        SUM(predicted_unit_sales) AS total_predicted_volume
    FROM read_parquet('{parquet_glob}')
""").df()

print("Production Forecast Distribution Summary (Holdout Replay):")
display(dist_stats.T.round(4))

print("\n[NOTE] Descriptive Volume Sanity:")
print(f"       Total Predicted Volume across Holdout: {dist_stats['total_predicted_volume'].iloc[0]:,.0f} units.")
print(f"       Mean Predicted Daily Demand per Store-SKU: {dist_stats['mean'].iloc[0]:.2f} units.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Production Forecast Distribution Summary (Holdout Replay):


,0
row_count,1.957197e+07
min_raw,-5.459010e+01
min_clipped,0.000000e+00
p5,1.954800e+00
p25,3.058100e+00
median,4.929200e+00
mean,9.179100e+00
p75,9.273000e+00
p95,2.821180e+01
p99,6.899170e+01



[NOTE] Descriptive Volume Sanity:
       Total Predicted Volume across Holdout: 179,653,444 units.
       Mean Predicted Daily Demand per Store-SKU: 9.18 units.


In [13]:
# Cell 14 — Operational Forecast Breakdown by Store & Product Family (Latest Slice)
op_file_glob = str(OP_FILE_PATH).replace("\\", "/")

store_op_summary = con.execute(f"""
    SELECT
        store_id,
        COUNT(*) AS active_skus,
        SUM(predicted_unit_sales) AS total_predicted_sales,
        AVG(predicted_unit_sales) AS avg_forecast_per_sku,
        SUM(operational_upper_80 - predicted_unit_sales) AS total_safety_buffer_units
    FROM read_parquet('{op_file_glob}')
    GROUP BY store_id
    ORDER BY total_predicted_sales DESC
    LIMIT 10
""").df()

print("Top 10 Stores by Predicted Daily Demand (Latest Operational Slice 2017-08-14):")
display(store_op_summary.round(2))

family_op_summary = con.execute(f"""
    SELECT
        family,
        COUNT(*) AS active_store_skus,
        SUM(predicted_unit_sales) AS total_predicted_sales,
        AVG(predicted_unit_sales) AS avg_forecast_per_sku,
        SUM(operational_upper_80 - predicted_unit_sales) AS total_safety_buffer_units
    FROM read_parquet('{op_file_glob}')
    GROUP BY family
    ORDER BY total_predicted_sales DESC
    LIMIT 10
""").df()

print("\nTop 10 Product Families by Predicted Daily Demand:")
display(family_op_summary.round(2))


Top 10 Stores by Predicted Daily Demand (Latest Operational Slice 2017-08-14):


,store_id,active_skus,total_predicted_sales,avg_forecast_per_sku,total_safety_buffer_units
0,44,2448,35890.88,14.66,18830.53
1,45,2491,32688.75,13.12,19186.04
2,3,2353,30401.98,12.92,16700.24
3,47,2449,28901.87,11.80,18141.37
4,49,2359,27383.08,11.61,16580.51
5,46,2312,22480.18,9.72,15817.59
6,48,2148,19253.32,8.96,14120.54
7,11,1694,18772.57,11.08,9926.35
8,51,1812,18522.45,10.22,11452.66
9,8,2099,17926.19,8.54,12342.84



Top 10 Product Families by Predicted Daily Demand:


,family,active_store_skus,total_predicted_sales,avg_forecast_per_sku,total_safety_buffer_units
0,GROCERY I,29571,208936.17,7.07,158157.79
1,BEVERAGES,13294,154643.36,11.63,80966.12
2,PRODUCE,7576,119611.38,15.79,54245.04
3,CLEANING,9516,51458.85,5.41,43788.21
4,DAIRY,6539,39026.72,5.97,33717.12
5,BREAD/BAKERY,3175,25575.19,8.06,17951.78
6,MEATS,1453,16232.99,11.17,9390.18
7,POULTRY,1116,15636.16,14.01,8451.77
8,DELI,2557,13859.15,5.42,12239.41
9,PERSONAL CARE,2685,13834.06,5.15,12506.50


In [14]:
# Cell 15 — Calibrated Uncertainty & Operational Bound Spread Analysis
parquet_glob = str(PROD_PRED_DIR / "forecast_date=*/part-*.parquet").replace("\\", "/")

tier_spread_summary = con.execute(f"""
    SELECT
        demand_tier,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS row_share_pct,
        AVG(predicted_unit_sales) AS mean_predicted_demand,
        AVG(operational_lower_80) AS mean_lower_bound,
        AVG(operational_upper_80) AS mean_upper_bound,
        AVG(operational_upper_80 - predicted_unit_sales) AS mean_upper_spread,
        AVG(expected_error_mae) AS mean_tier_mae
    FROM read_parquet('{parquet_glob}')
    GROUP BY demand_tier
    ORDER BY demand_tier
""").df()

print("Calibrated Empirical Uncertainty Bounds Spread across Demand Tiers:")
display(tier_spread_summary.round(4))


Calibrated Empirical Uncertainty Bounds Spread across Demand Tiers:


,demand_tier,row_count,row_share_pct,mean_predicted_demand,mean_lower_bound,mean_upper_bound,mean_upper_spread,mean_tier_mae
0,1_Low,7154798,36.56,2.9329,0.8729,5.2637,2.3309,1.6434
1,2_Mid,6338980,32.39,5.5449,1.6754,9.9284,4.3835,2.9267
2,3_High,6078188,31.06,20.3220,10.4237,30.6873,10.3654,8.2584


In [15]:
# Cell 16 — Confirm Physical Dataset Structure
partition_dirs = list(PROD_PRED_DIR.glob("forecast_date=*"))
total_part_size = sum(f.stat().st_size for f in PROD_PRED_DIR.rglob("*.parquet"))

print(f"[OK] Physical Storage Verification:")
print(f"     Root Directory      : {PROD_PRED_DIR}")
print(f"     Date Partitions     : {len(partition_dirs)} folders")
print(f"     Operational File    : {OP_FILE_PATH.name} ({OP_FILE_PATH.stat().st_size:,} bytes)")
print(f"     Total Disk Footprint: {total_part_size:,} bytes ({total_part_size / (1024**2):.1f} MB)")


[OK] Physical Storage Verification:
     Root Directory      : D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\production_predictions
     Date Partitions     : 227 folders
     Operational File    : operational_simulation_latest.parquet (2,129,891 bytes)
     Total Disk Footprint: 505,167,348 bytes (481.8 MB)


In [16]:
# Cell 17 — Generate Scoring Run Contract (scoring_run_contract.json)
scoring_contract = {
    "schema_version": 1,
    "produced_by": "06_production_scoring_pipeline",
    "scoring_run_id": SCORING_RUN_ID,
    "scoring_timestamp": SCORING_TIMESTAMP,
    "model_metadata": {
        "model_version": "LightGBM_v1",
        "final_model_run_id": FINAL_MODEL_RUN_ID,
        "model_sha256": ACTUAL_SHA256,
        "n_features": 24,
        "n_trees": 496,
        "feature_names": FEATURES
    },
    "governance": {
        "retrained": RETRAINED,
        "retuned": RETUNED,
        "model_selection_performed": MODEL_SELECTION_PERFORMED,
        "production_scoring_run": PRODUCTION_SCORING_RUN,
        "upstream_nb05_blockers": 0
    },
    "scoring_modes": {
        "holdout_replay": {
            "mode_name": "HOLDOUT_REPLAY",
            "dates_scored": len(manifest_records),
            "total_rows": total_rows_scored,
            "min_forecast_date": str(manifest_records[0]["forecast_date"]),
            "max_forecast_date": str(manifest_records[-1]["forecast_date"]),
            "storage_format": "partitioned_parquet",
            "partition_pattern": "forecast_date=YYYY-MM-DD/part-00000.parquet"
        },
        "operational_simulation": {
            "mode_name": "OPERATIONAL_SIMULATION",
            "forecast_date": "2017-08-14",
            "target_date": "2017-08-15",
            "active_store_skus": len(op_scored_df),
            "file_path": str(OP_FILE_PATH)
        }
    },
    "uncertainty_calibration": {
        "source": "validation_residuals",
        "nominal_coverage": 0.80,
        "tier_thresholds": {
            "q33": q33,
            "q66": q66
        },
        "tier_quantiles": TIER_CALIBRATION
    },
    "quality_gates": {
        "zero_nan": True,
        "zero_inf": True,
        "zero_duplicates": True,
        "zero_negative_predictions": True,
        "d_to_d_plus_1_valid": True,
        "feature_chain_matched": True,
        "model_sha256_matched": True
    },
    "outputs": {
        "directory": str(PROD_PRED_DIR),
        "partitioned_dataset": str(PROD_PRED_DIR / "forecast_date=*/part-*.parquet"),
        "operational_file": str(OP_FILE_PATH),
        "scoring_contract": str(PROD_PRED_DIR / "scoring_run_contract.json"),
        "scoring_manifest": str(PROD_PRED_DIR / "scoring_manifest.json")
    }
}

CONTRACT_OUT_PATH = PROD_PRED_DIR / "scoring_run_contract.json"
with open(CONTRACT_OUT_PATH, "w") as f:
    json.dump(scoring_contract, f, indent=2)

print(f"[OK] Saved Scoring Run Contract to: {CONTRACT_OUT_PATH}")


[OK] Saved Scoring Run Contract to: D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\production_predictions\scoring_run_contract.json

In [17]:
# Cell 18 — Generate Scoring Manifest (scoring_manifest.json)
scoring_manifest = {
    "scoring_run_id": SCORING_RUN_ID,
    "scoring_timestamp": SCORING_TIMESTAMP,
    "total_partitions": len(manifest_records),
    "total_rows": total_rows_scored,
    "partitions": manifest_records
}

MANIFEST_OUT_PATH = PROD_PRED_DIR / "scoring_manifest.json"
with open(MANIFEST_OUT_PATH, "w") as f:
    json.dump(scoring_manifest, f, indent=2)

print(f"[OK] Saved Scoring Manifest to: {MANIFEST_OUT_PATH} ({len(manifest_records)} partition records)")


[OK] Saved Scoring Manifest to: D:\Demand_forecasting\inventory-lakehouse\data_lake\gold\production_predictions\scoring_manifest.json (227 partition records)


In [18]:
# Cell 19 — Post-Write Verification & Read-Back Gate
print("[CHECK] Executing Final Post-Write Read-Back Gate...")

parquet_glob = str(PROD_PRED_DIR / "forecast_date=*/part-*.parquet").replace("\\", "/")

# Read back sample from partitioned dataset
sample_readback = con.execute(f"""
    SELECT * 
    FROM read_parquet('{parquet_glob}')
    LIMIT 1000
""").df()

assert len(sample_readback) == 1000, "[FAIL] Sample readback failed"
assert set(OUTPUT_COLUMNS).issubset(set(sample_readback.columns)), f"[FAIL] Schema columns mismatch in readback: {sample_readback.columns}"

# Hard File Assertions
assert (PROD_PRED_DIR / "scoring_run_contract.json").exists()
assert (PROD_PRED_DIR / "scoring_manifest.json").exists()
assert OP_FILE_PATH.exists()
assert len(list(PROD_PRED_DIR.glob("forecast_date=*"))) == 227

print(f"[PASS] All Production Scoring Artifacts Successfully Verified on Disk:")
print(f"       - 227 Forecast Date Partitions ({total_rows_scored:,} rows)")
print(f"       - 1 Standalone Operational File ({len(op_scored_df):,} rows)")
print(f"       - 1 Scoring Run Contract JSON")
print(f"       - 1 Scoring Manifest JSON")


[CHECK] Executing Final Post-Write Read-Back Gate...
[PASS] All Production Scoring Artifacts Successfully Verified on Disk:
       - 227 Forecast Date Partitions (19,571,966 rows)
       - 1 Standalone Operational File (84,659 rows)
       - 1 Scoring Run Contract JSON
       - 1 Scoring Manifest JSON


## 6. Production Scoring Complete & Downstream Hand-Off

### Summary of Completed Scoring Operations
1. **Model Governance & Integrity:**
   - Evaluated frozen LightGBM booster (`lightgbm_model.txt`, SHA-256: `ea408d13e01a...`, Run ID: `e8da7be6...`) with **zero retraining, zero tuning, and zero target leakage**.
   - Validated upstream NB05 readiness gate with 0 blocking issues.
2. **Scored Datasets Generated:**
   - **Mode A (`HOLDOUT_REPLAY`):** 19,571,966 rows partitioned by `forecast_date` across all 227 holdout dates in `data_lake/gold/production_predictions/`.
   - **Mode B (`OPERATIONAL_SIMULATION`):** 84,659 store-SKU forecasts for the latest operational date origin ($D = \text{2017-08-14} \rightarrow D+1 = \text{2017-08-15}$) in `operational_simulation_latest.parquet`.
3. **Calibrated Empirical Uncertainty:**
   - Attached non-parametric $q_{10}$ and $q_{90}$ empirical residual offsets by demand tier (`operational_lower_80`, `operational_upper_80`).

---

### Downstream Contract for Notebook 07 (Newsvendor Inventory Optimization)
Notebook 07 directly consumes `operational_simulation_latest.parquet` and `production_predictions/` to compute optimal replenishment quantities:

| Column | Role in Notebook 07 |
|---|---|
| `predicted_unit_sales` | Mean expected demand $\mu_d$ for day $D+1$. |
| `operational_upper_80` | High-service-level demand quantile for safety buffer planning. |
| `expected_error_mae` | Empirical standard deviation proxy for continuous order optimization. |
| `demand_tier` | Segmentation tag for applying business service level policies ($SL_{\text{High}} > SL_{\text{Low}}$). |

> **Strict Division of Responsibility:**
> Notebook 06 provides the forecast $\hat{y}$ and empirical uncertainty bounds. Notebook 07 decides $C_u$, $C_o$, service level targets, lead times, safety stocks, and Newsvendor order quantities ($Q^*$).
